# Part III walkthrough — the forward problem

Parts I and II built a toolkit on a problem chosen for its docility. Here we point it
at the incompressible Navier–Stokes equations, where all three of the toolkit's weak
points are provoked at once:

$$\partial_t \mathbf{v} + (\mathbf{v}\cdot\nabla)\mathbf{v}
  = -\tfrac{1}{\rho}\nabla p + \nu\nabla^2\mathbf{v},
  \qquad \nabla\cdot\mathbf{v} = 0$$

1. **nonlinear advection** — products of trains, so ranks multiply;
2. **a nonlocal pressure constraint** — a global elliptic solve every step;
3. **boundary conditions on complex geometry** — dealt with in Part II.

From here on we use `quimb` for the representation and compression, and keep our own
solver from Part II. **That seam is itself the lesson:** no tensor-network library
ships a linear solver for $Ax=b$, because the quantum-physics use case wants ground
states instead.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_cfd as cfd
import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 10})

## 1. Why pressure is the hard part

Pressure is not dynamical. There is no evolution equation for it; it is whatever
enforces $\nabla\cdot\mathbf{v}=0$ instantly. Taking the divergence of the momentum
equation gives an elliptic problem that couples the whole domain at once — and that is
where the runtime goes, classical solver or tensor one.

A detail that is easy to get wrong and expensive to debug: the pressure operator must be
$\mathrm{div}(\mathrm{grad})$ built from **the same** difference operators used for the
gradient and the divergence. Substitute the compact three-point Laplacian and the
projection stops projecting.

In [ ]:
n = 6
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / 2 ** n, chi_max=32, cutoff=1e-7, poisson_sweeps=2)
print(f"grid                 : {2**n} x {2**n} = {4**n:,d} cells, {2*n} binary sites")
print(f"d/dx, d/dy  MPO rank : {max(tn.mpo_ranks(flow.Dx))}")
print(f"Laplacian   MPO rank : {max(tn.mpo_ranks(flow.L))}")
print(f"div(grad)   MPO rank : {max(tn.mpo_ranks(flow.Lp))}")

## 2. One Chorin step, watched closely

Start from a vortex pair. The initial field is built with `np.gradient`, whose stencil
is *not* the one our operators use, so it starts off visibly non-solenoidal — which
makes the projection easy to see working.

In [ ]:
N = 2 ** n
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
psi = 0.05 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)
u = qq.from_grid(np.gradient(psi, xx, axis=1), eps=1e-9)
v = qq.from_grid(-np.gradient(psi, xx, axis=0), eps=1e-9)

print(f"before the first step: |div v| = {tn.tt_norm(cfd.divergence(flow, u, v)):.3e}")
u, v, p = flow.step(u, v)
print(f"after  the first step: |div v| = {tn.tt_norm(cfd.divergence(flow, u, v)):.3e}")
print(f"\npressure train: chi = {max(tn.tt_ranks(p))}")

Three orders of magnitude, from one linear solve. That solve is the only global
operation in the timestep, and everything else is local arithmetic plus rounding.

## 3. Where the time actually goes

In [ ]:
flow = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=32, cutoff=1e-7, poisson_sweeps=1)
u = qq.from_grid(np.gradient(psi, xx, axis=1), eps=1e-9)
v = qq.from_grid(-np.gradient(psi, xx, axis=0), eps=1e-9)

energy, divergence_hist, frames = [], [], {}
t0 = time.perf_counter()
n_steps = 40
for k in range(n_steps):
    u, v, p = flow.step(u, v)
    energy.append(cfd.energy(u, v))
    divergence_hist.append(tn.tt_norm(cfd.divergence(flow, u, v)))
    if k in (0, n_steps // 2, n_steps - 1):
        frames[k] = qq.to_grid(u)
wall = time.perf_counter() - t0

print(f"{n_steps} steps in {wall:.1f} s  ({wall / n_steps:.2f} s/step)\n")
for stage, t in flow.timings.items():
    print(f"  {stage:>11}: {t:>6.2f} s  ({100 * t / sum(flow.timings.values()):>5.1f}%)")
print(f"\nThe pressure Poisson solve is {100 * flow.poisson_share():.0f}% of the runtime.")
print("Peddinti et al. (2024) report 80.1% for the full 3D solver. Same story.")

That number is the single most useful thing to know about this method. Any effort spent
anywhere other than the elliptic solve is, to first order, wasted — which is why
preconditioning in tensor-train format is the open problem with the largest payoff.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
for a, (k, f) in zip(ax, frames.items()):
    im = a.imshow(f.T, origin="lower", cmap="RdBu_r")
    a.set_title(f"$u$ at step {k}")
    a.set_xticks([]), a.set_yticks([])
ax[3].semilogy(divergence_hist)
ax[3].set(xlabel="step", ylabel=r"$\|\nabla\cdot\mathbf{v}\|$",
          title="the constraint, over time")
plt.tight_layout()

### Truncation does not respect physics

Watch the divergence creep back up. SVD truncation is optimal in the Frobenius norm,
and the Frobenius norm knows nothing about $\nabla\cdot\mathbf{v}=0$. Every rounding
step puts a little divergence back, and the projection removes it again, and the balance
between them is set by `chi_max`.

In [ ]:
for chi in (8, 16, 32, 48):
    f2 = cfd.Flow(n, nu=2e-3, dt=0.4 / N, chi_max=chi, cutoff=1e-9, poisson_sweeps=1)
    uu = qq.from_grid(np.gradient(psi, xx, axis=1), eps=1e-9)
    vv = qq.from_grid(-np.gradient(psi, xx, axis=0), eps=1e-9)
    for _ in range(20):
        uu, vv, _ = f2.step(uu, vv)
    print(f"chi_max = {chi:>2}: after 20 steps  |div v| = "
          f"{tn.tt_norm(cfd.divergence(f2, uu, vv)):.2e},  "
          f"E = {cfd.energy(uu, vv):.4f}")

Structure-preserving rounding — truncation that keeps mass, energy or the divergence
constraint — is genuinely open. For now, `chi_max` is the knob and the divergence is
the diagnostic you watch.

## 4. Reading out without decompressing

If you have to rebuild the $2^{2n}$ array to look at the answer, the compression bought
nothing. Two readouts cost far less:

* a **single point**, by contracting one matrix chain: $O(n\chi^2)$;
* a **coarse-grained field**, by contracting the finest bits against $(1,1)/2$, which
  is exactly average pooling: $O(n\chi^3)$.

In [ ]:
print(f"{'n':>3} {'cells':>12} {'chi':>5} {'1 point (us)':>14} {'16x16 field (ms)':>18} "
      f"{'full field (ms)':>17}")
for ni in (5, 6, 7, 8, 9, 10):
    Ni = 2 ** ni
    xi = np.linspace(0, 1, Ni, endpoint=False)
    Xi, Yi = np.meshgrid(xi, xi, indexing="ij")
    fi = np.exp(-30 * ((Xi - 0.4) ** 2 + (Yi - 0.6) ** 2)) * np.sin(6 * np.pi * Xi * Yi)
    ci = qq.from_grid(fi, eps=1e-8)
    bits = [1, 0] * ni

    t0 = time.perf_counter()
    for _ in range(2000):
        qq.evaluate(ci, bits)
    t_point = (time.perf_counter() - t0) / 2000 * 1e6

    t0 = time.perf_counter()
    for _ in range(10):
        qq.to_grid(qq.coarse_grain(qq.to_mps(ci), 4))
    t_coarse = (time.perf_counter() - t0) / 10 * 1e3

    t0 = time.perf_counter()
    for _ in range(10):
        qq.to_grid(ci)
    t_full = (time.perf_counter() - t0) / 10 * 1e3

    print(f"{ni:>3} {Ni**2:>12,d} {max(tn.tt_ranks(ci)):>5} {t_point:>14.1f} "
          f"{t_coarse:>18.2f} {t_full:>17.2f}")

The first two columns of timings barely move as the grid grows by a factor of a
thousand; the last one tracks the cell count. Reconstructing the dense field is the one
operation whose cost the compression did nothing about — so build the diagnostics you
need (slices, averages, spectra, single probes) to work on the train directly.

In [ ]:
uc = u if isinstance(u, list) else qq.cores(u)
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(qq.to_grid(qq.coarse_grain(qq.to_mps(uc), 4)).T, origin="lower", cmap="RdBu_r")
ax[0].set_title("coarse-grained, 16x16")
ax[1].imshow(qq.to_grid(uc).T, origin="lower", cmap="RdBu_r")
ax[1].set_title(f"full field, {N}x{N}")
for a in ax:
    a.set_xticks([]), a.set_yticks([])
plt.tight_layout()

## 5. Cost against resolution

The claim is polylogarithmic in the number of cells. Measure it: fix everything except
$n$ and time one timestep.

In [ ]:
print(f"{'n':>3} {'cells':>12} {'chi':>5} {'s / step':>10}")
for ni in (5, 6, 7, 8):
    Ni = 2 ** ni
    xi = np.linspace(0, 1, Ni, endpoint=False)
    Xi, Yi = np.meshgrid(xi, xi, indexing="ij")
    psi_i = 0.05 * np.sin(2 * np.pi * Xi) * np.sin(2 * np.pi * Yi)
    fi = cfd.Flow(ni, nu=2e-3, dt=0.4 / Ni, chi_max=24, cutoff=1e-7, poisson_sweeps=1)
    ui = qq.from_grid(np.gradient(psi_i, xi, axis=1), eps=1e-9)
    vi = qq.from_grid(-np.gradient(psi_i, xi, axis=0), eps=1e-9)
    ui, vi, _ = fi.step(ui, vi)                       # warm up the pressure guess
    t0 = time.perf_counter()
    for _ in range(3):
        ui, vi, _ = fi.step(ui, vi)
    dt_step = (time.perf_counter() - t0) / 3
    print(f"{ni:>3} {Ni**2:>12,d} {fi.rank_history[-1]:>5} {dt_step:>10.2f}")

Read that table carefully, because it is the place where an enthusiast would overclaim.

The cell count grows by a factor of 64 across those rows. The runtime grows by roughly
a factor of 20 — sublinear in the number of cells, which is the point, but well above
the $O(n)$ that the scaling argument promises. Two reasons, and neither is physics:

* this is unoptimised Python, and the per-site constant is dominated by interpreter
  overhead and by the local conjugate-gradient solves, not by the tensor algebra;
* the bond dimension is capped at the same value at every resolution, so the *rank*
  contribution is flat, but the number of sites and the number of CG iterations both
  grow.

The measured exponent in the production solver of Peddinti *et al.* (2024) is
$n\chi^{4.1}$ against a worst case of $n\chi^{6}$. The lesson to take from this table is
the shape of the curve, not its constant: 64 times the cells for 20 times the work is
already a different regime from classical CFD, and a tuned implementation moves it
further.

## 6. Expressivity: the question everything hinges on

Every result so far assumed $\chi$ stays small. When is that true? There are two
separate questions with different answers:

* $\chi$ versus **resolution** at fixed physics — benign, that is the quantics promise
  and we measured it in Part I;
* $\chi$ versus **Reynolds number** — the real question.

We cannot run a $\mathrm{Re}_\lambda = 315$ DNS here. What we can do is build synthetic
fields with a widening inertial range and watch the rank needed to hold them.

In [ ]:
m = 8
Nt = 2 ** m
rng = np.random.default_rng(7)
kf = np.fft.fftfreq(Nt) * Nt
KX, KY = np.meshgrid(kf, kf, indexing="ij")
K = np.hypot(KX, KY)
K[0, 0] = 1.0
phase = rng.standard_normal((Nt, Nt)) + 1j * rng.standard_normal((Nt, Nt))

print(f"{'k_max (inertial range)':>24} {'chi @ 1e-2':>12} {'chi @ 1e-3':>12}")
for kmax in (4, 8, 16, 32, 64):
    spec = K ** (-5 / 6)
    spec[K > kmax] = 0.0
    field = np.real(np.fft.ifft2(spec * phase))
    field /= field.std()
    c2 = max(tn.tt_ranks(qq.from_grid(field, eps=1e-2)))
    c3 = max(tn.tt_ranks(qq.from_grid(field, eps=1e-3)))
    print(f"{kmax:>24} {c2:>12} {c3:>12}")

The rank climbs with the width of the inertial range, not with the grid. That is the
honest shape of the claim:

* **supported** — turbulence compresses well at moderate Reynolds number, and
  Pisoni *et al.* (2026) show four orders of magnitude of compression at
  $\mathrm{Re}_\lambda = 315$ on $1024^3$ with the spectrum essentially intact;
* **not supported** — low rank at arbitrary Reynolds number and every scale.

And note which diagnostic you use. The energy spectrum is the easy one; it looks fine
long after the flow does not. The demanding diagnostics are the probability
distributions of velocity increments and the flatness, and those are what separate the
bit orderings.

---
### What this demonstrator is not

Boundary conditions here are homogeneous Dirichlet on every wall, inherited from the
difference operators. There is no ghost-cell inlet/outlet treatment and no immersed
geometry — those are what turn this into the solver of Peddinti *et al.* (2024), and
they are the difference between a teaching demonstrator and a validated code.

What this *does* reproduce faithfully is the cost structure, which is the part worth
carrying away: the Poisson solve dominates, rounding fights the constraint, and the
rank is the thing to watch.